# Step 4 - driving results

Hold each gene, and then each gene pair, at its observed start and end value
and integrate the rest of the network forward.

In [ ]:
import os
import sys

NETDESDUO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "neutrophil_data"))
sys.path.insert(0, NETDESDUO_ROOT)
import NetDesDuo
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import random
import importlib
import math
import joblib
from sklearn.metrics import mean_squared_error
importlib.reload(NetDesDuo)
import math
import matplotlib.pyplot as plt

# fitting run tree - not committed, edit for your setup
WORK_DIR = "/projects/lulab/alex/neutrophil_case/network_optimization"
os.chdir(WORK_DIR)

## Naive

In [ ]:
tf11=pd.read_csv('naive_results/naive_expression_data.csv')
genes = pd.read_csv('naive_results/naive_genes_expressed.csv')
genes = genes['x'].to_list()
tf11 = tf11.transpose()
tf11 = tf11[1:100000]
tf11.index = genes

In [ ]:
#restore the average pseudotime values to before log-ing
tf12expression = tf11.apply(lambda x: (math.e**x - 1))
tf12expression[tf12expression <= 0.001] = 0.001
#scale expression
tf12_scaled = tf12expression.div(tf12expression.max(axis=1),axis=0)
tf12expression_logtarget = tf12_scaled.apply(lambda x: np.log2(x))
tf12expression_logtarget = tf12expression_logtarget.dropna()
tf12expression = tf12expression.dropna()

In [ ]:
network0 = pd.read_csv('naive_results/naive_initial_network.csv')
network1 = network0[["Source", "Target", "Interaction"]]
gene_list = list(pd.unique(network1['Target']))
network1.columns = ['Source', 'Target', 'Interaction']
ids = list(range(len(gene_list)))

In [ ]:
ob_newall = joblib.load("naive_results/naive_networks/n_0.5_k_1.5/naive_ob_newall_signed.joblib")
res_final = joblib.load("naive_results/naive_networks/n_0.5_k_1.5/naive_res_final_signed.joblib")
pseudotime_pick = pd.read_csv('naive_results/pseudotime_pick.csv')[['x']]

## Drive every gene and every gene pair

In [ ]:
drivers = gene_list
tf12expression_logtarget2, tf12expression_target2, gene_position,gene_list2 = NetDesDuo.process_gene_expression(tf12expression_logtarget, 
                                                                                                             tf12expression, 
                                                                                                             gene_list, 
                                                                                                             ob_newall)
ob_newall2, res_final2,gene_position2=NetDesDuo.Fit_withoutinput(gene_list=gene_list2, 
                                                              res_final=res_final,
                                                              ob_newall=ob_newall,
                                                              gene_position=gene_position,
                                                              tfexpression_logtarget=tf12expression_logtarget)

In [ ]:
pairs = []
for i in range(len(drivers)):
    for j in range(i + 1): 
        pairs.append([i, j])

In [ ]:
cache_naive = joblib.load("naive_driving_results/cache_naive.joblib")
def drive (pair, tf12expression_target2=tf12expression_target2, cache_naive=cache_naive):
    i = pair[0]
    j = pair[1]
    if i == j:
        file_name = "naive_driving_results/one_gene_driving/" + "single_" + gene_list[i] + ".pdf"
        print(file_name)
        dri_genes = [gene_list[i]]
    else:
        sorted_name = sorted((gene_list[i], gene_list[j]))
        file_name = "naive_driving_results/two_gene_driving/" + sorted_name[0] + "_" + sorted_name[1] + ".pdf"
        print(file_name)
        dri_genes = list(sorted_name)
        
    df3, df4,cache_naive,figs = NetDesDuo.driving_results(
        dri_genes= dri_genes,
        gene_list=gene_list,
        tfexpression_target2=tf12expression_target2,
        network=network1,
        tfexpression=tf12expression,
        pseudotime_pick=pseudotime_pick,
        res_final=res_final2,
        ob_newall=ob_newall2,
        gene_position=gene_position2,
        cached = True,
        cache = cache_naive)
    figs.savefig(file_name, dpi=300, bbox_inches='tight')
    plt.close(figs)
    return cache_naive

In [ ]:
results = joblib.Parallel(n_jobs=12, verbose=100, batch_size=1)(
    joblib.delayed(drive)(pair=pair) for pair in pairs
)

cache_naive = {}
for partial in results:
    cache_naive.update(partial)

joblib.dump(cache_naive, "naive_driving_results/cache_naive.joblib")

In [ ]:
cache_naive = joblib.load("naive_driving_results/cache_naive.joblib")

#TEMPORARY
f1, b1, dif1 = NetDesDuo.one_gene_driving(
    dt=1, t_tot=50000, gene_list=np.array(gene_list), drivers=[],
    tfexpression_target2=tf12expression_target2, res_final=res_final2,
    gene_position=gene_position2, network=network1, tfexpression=tf12expression,
    ob_all=ob_newall2, cached=True, cache=cache_naive)

f2, b2, dif2 = NetDesDuo.two_gene_driving(
    dt=1, t_tot=50000, gene_list=np.array(gene_list), drivers=[],
    tfexpression_target2=tf12expression_target2, res_final=res_final2,
    gene_position=gene_position2, tfexpression=tf12expression,
    ob_all=ob_newall2, network=network1, cached=True, cache=cache_naive)

In [ ]:
joblib.dump(f1, "naive_driving_results/naive_f1.joblib")
joblib.dump(b1, "naive_driving_results/naive_b1.joblib")
joblib.dump(dif1, "naive_driving_results/naive_dif1.joblib")
joblib.dump(f2, "naive_driving_results/naive_f2.joblib")
joblib.dump(b2, "naive_driving_results/naive_b2.joblib")
joblib.dump(dif2, "naive_driving_results/naive_dif2.joblib")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

plt.close("all")

# -------------------------
# f2 vs dif2: edge deletions
# -------------------------
f2_tmp = f2[["gene1", "gene2", "position1", "position2", "MSE"]].rename(
    columns={"MSE": "mse_f"}
)

dif2_tmp = dif2[["gene1", "gene2", "position1", "position2", "MSE"]].rename(
    columns={"MSE": "mse_dif"}
)

edge_df = f2_tmp.merge(
    dif2_tmp,
    on=["gene1", "gene2", "position1", "position2"],
    how="inner"
)

edge_df = edge_df[["mse_f", "mse_dif"]]

# -------------------------
# f1 vs dif1: gene deletions
# -------------------------
f1_tmp = f1[["gene", "MSE"]].rename(columns={"MSE": "mse_f"})
dif1_tmp = dif1[["gene", "MSE"]].rename(columns={"MSE": "mse_dif"})

gene_df = f1_tmp.merge(
    dif1_tmp,
    on="gene",
    how="inner"
)

gene_df = gene_df[["mse_f", "mse_dif"]]

# -------------------------
# combine both
# -------------------------
plot_df = pd.concat([edge_df, gene_df], ignore_index=True)

# remove failed/error rows and invalid log values
plot_df = plot_df[
    (plot_df["mse_f"] > 0) &
    (plot_df["mse_dif"] > 0) &
    (plot_df["mse_f"] <= 10000) &
    (plot_df["mse_dif"] <= 10000)
].copy()

# -------------------------
# plot
# -------------------------
lo = min(plot_df["mse_f"].min(), plot_df["mse_dif"].min())
hi = max(plot_df["mse_f"].max(), plot_df["mse_dif"].max())

plt.figure(figsize=(7, 7))

plt.scatter(
    plot_df["mse_f"],
    plot_df["mse_dif"],
    color="tab:blue",
    alpha=0.7
)

plt.plot(
    [lo, hi],
    [lo, hi],
    linestyle="--",
    linewidth=1,
    color="black"
)

plt.xscale("log")
plt.yscale("log")

plt.xlim(lo * 0.7, hi * 1.3)
plt.ylim(lo * 0.7, hi * 1.3)

plt.xlabel("MSE (forward driving vs experiment)")
plt.ylabel("MSE (forward driving vs backwards driving)")
plt.title("MSE comparison")

plt.tight_layout()
plt.savefig("naive_driving_results/MSE_plot.pdf")
plt.show()

## Tumour-bearing

In [ ]:
#tf11=pd.read_csv('/Users/alexren/desktop/Neutrophil/cancer_expression_data.csv')
tf11=pd.read_csv('cancer_results/cancer_expression_data.csv')
#genes = pd.read_csv('/Users/alexren/desktop/Neutrophil/cancer_genes_expressed.csv')
genes = pd.read_csv('cancer_results/cancer_genes_expressed.csv')
genes = genes['x'].to_list()
tf11 = tf11.transpose()
tf11 = tf11[1:100000]
tf11.index = genes

In [ ]:
#restore the average pseudotime values to before log-ing
tf12expression = tf11.apply(lambda x: (math.e**x - 1))
tf12expression[tf12expression <= 0.001] = 0.001
#scale expression
tf12_scaled = tf12expression.div(tf12expression.max(axis=1),axis=0)
tf12expression_logtarget = tf12_scaled.apply(lambda x: np.log2(x))
tf12expression_logtarget = tf12expression_logtarget.dropna()
tf12expression = tf12expression.dropna()

In [ ]:
network0 = pd.read_csv('cancer_results/cancer_initial_network.csv')
network1 = network0[["Source", "Target", "Interaction"]]
gene_list = list(pd.unique(network1['Target']))
network1.columns = ['Source', 'Target', 'Interaction']
ids = list(range(len(gene_list)))

In [ ]:
ob_newall = joblib.load("cancer_results/cancer_networks/n_0.5_k_1.5/cancer_ob_newall_signed.joblib")
res_final = joblib.load("cancer_results/cancer_networks/n_0.5_k_1.5/cancer_res_final_signed.joblib")
pseudotime_pick = pd.read_csv('cancer_results/pseudotime_pick.csv')[['x']]

In [ ]:
drivers = gene_list
tf12expression_logtarget2, tf12expression_target2, gene_position,gene_list2 = NetDesDuo.process_gene_expression(tf12expression_logtarget, 
                                                                                                             tf12expression, 
                                                                                                             gene_list, 
                                                                                                             ob_newall)
ob_newall2, res_final2,gene_position2=NetDesDuo.Fit_withoutinput(gene_list=gene_list2, 
                                                              res_final=res_final,
                                                              ob_newall=ob_newall,
                                                              gene_position=gene_position,
                                                              tfexpression_logtarget=tf12expression_logtarget)

In [ ]:
pairs = []
for i in range(len(drivers)):
    for j in range(i + 1): 
        pairs.append([i, j])

In [ ]:
cache_cancer = joblib.load("cancer_driving_results/cache_cancer.joblib")
def drive (pair, tf12expression_target2=tf12expression_target2, cache_cancer=cache_cancer):
    i = pair[0]
    j = pair[1]
    if i == j:
        file_name = "cancer_driving_results/one_gene_driving/" + "single_" + gene_list[i] + ".pdf"
        print(file_name)
        dri_genes = [gene_list[i]]
    else:
        sorted_name = sorted((gene_list[i], gene_list[j]))
        file_name = "cancer_driving_results/two_gene_driving/" + sorted_name[0] + "_" + sorted_name[1] + ".pdf"
        print(file_name)
        dri_genes = list(sorted_name)
        
    df3, df4,cache_cancer,figs = NetDesDuo.driving_results(
        dri_genes= dri_genes,
        gene_list=gene_list,
        tfexpression_target2=tf12expression_target2,
        network=network1,
        tfexpression=tf12expression,
        pseudotime_pick=pseudotime_pick,
        res_final=res_final2,
        ob_newall=ob_newall2,
        gene_position=gene_position2,
        cached = True,
        cache = cache_cancer)
    figs.savefig(file_name, dpi=300, bbox_inches='tight')
    plt.close(figs)
    return cache_cancer

In [ ]:
results = joblib.Parallel(n_jobs=12, verbose=100, batch_size=1)(
    joblib.delayed(drive)(pair=pair) for pair in pairs
)

for partial in results:
    cache_cancer.update(partial)

joblib.dump(cache_cancer, "cancer_driving_results/cache_cancer.joblib")

In [ ]:
cache_cancer = joblib.load("cancer_driving_results/cache_cancer.joblib")

#TEMPORARY
f1, b1, dif1 = NetDesDuo.one_gene_driving(
    dt=1, t_tot=50000, gene_list=np.array(gene_list) , drivers=[],
    tfexpression_target2=tf12expression_target2, res_final=res_final2,
    gene_position=gene_position2, network=network1, tfexpression=tf12expression,
    ob_all=ob_newall2, cached=True, cache=cache_cancer)

f2, b2, dif2 = NetDesDuo.two_gene_driving(
    dt=1, t_tot=50000, gene_list=np.array(gene_list) , drivers=[],
    tfexpression_target2=tf12expression_target2, res_final=res_final2,
    gene_position=gene_position2, tfexpression=tf12expression,
    ob_all=ob_newall2, network=network1, cached=True, cache=cache_cancer)

In [ ]:
joblib.dump(f1, "cancer_driving_results/cancer_f1.joblib")
joblib.dump(b1, "cancer_driving_results/cancer_b1.joblib")
joblib.dump(dif1, "cancer_driving_results/cancer_dif1.joblib")
joblib.dump(f2, "cancer_driving_results/cancer_f2.joblib")
joblib.dump(b2, "cancer_driving_results/cancer_b2.joblib")
joblib.dump(dif2, "cancer_driving_results/cancer_dif2.joblib")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

plt.close("all")

# -------------------------
# f2 vs dif2: edge deletions
# -------------------------
f2_tmp = f2[["gene1", "gene2", "position1", "position2", "MSE"]].rename(
    columns={"MSE": "mse_f"}
)

dif2_tmp = dif2[["gene1", "gene2", "position1", "position2", "MSE"]].rename(
    columns={"MSE": "mse_dif"}
)

edge_df = f2_tmp.merge(
    dif2_tmp,
    on=["gene1", "gene2", "position1", "position2"],
    how="inner"
)

edge_df = edge_df[["mse_f", "mse_dif"]]

# -------------------------
# f1 vs dif1: gene deletions
# -------------------------
f1_tmp = f1[["gene", "MSE"]].rename(columns={"MSE": "mse_f"})
dif1_tmp = dif1[["gene", "MSE"]].rename(columns={"MSE": "mse_dif"})

gene_df = f1_tmp.merge(
    dif1_tmp,
    on="gene",
    how="inner"
)

gene_df = gene_df[["mse_f", "mse_dif"]]

# -------------------------
# combine both
# -------------------------
plot_df = pd.concat([edge_df, gene_df], ignore_index=True)

# remove failed/error rows and invalid log values
plot_df = plot_df[
    (plot_df["mse_f"] > 0) &
    (plot_df["mse_dif"] > 0) &
    (plot_df["mse_f"] <= 10000) &
    (plot_df["mse_dif"] <= 10000)
].copy()

# -------------------------
# plot
# -------------------------
lo = min(plot_df["mse_f"].min(), plot_df["mse_dif"].min())
hi = max(plot_df["mse_f"].max(), plot_df["mse_dif"].max())

plt.figure(figsize=(7, 7))

plt.scatter(
    plot_df["mse_f"],
    plot_df["mse_dif"],
    color="tab:blue",
    alpha=0.7
)

plt.plot(
    [lo, hi],
    [lo, hi],
    linestyle="--",
    linewidth=1,
    color="black"
)

plt.xscale("log")
plt.yscale("log")

plt.xlim(lo * 0.7, hi * 1.3)
plt.ylim(lo * 0.7, hi * 1.3)

plt.xlabel("MSE (forward driving vs experiment)")
plt.ylabel("MSE (forward driving vs backwards driving)")
plt.title("MSE comparison")

plt.tight_layout()
plt.savefig("cancer_driving_results/MSE_plot.pdf")
plt.show()

## Both conditions on one pair of axes (figure 6D)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

# =========================================================
# INPUTS: change only these names if your objects differ
# =========================================================
NAIVE_F2   = joblib.load("naive_driving_results/naive_f2.joblib")
NAIVE_DIF2 = joblib.load("naive_driving_results/naive_dif2.joblib")
NAIVE_F1   = joblib.load("naive_driving_results/naive_f1.joblib")
NAIVE_DIF1 = joblib.load("naive_driving_results/naive_dif1.joblib")

CANCER_F2   = joblib.load("cancer_driving_results/cancer_f2.joblib")
CANCER_DIF2 = joblib.load("cancer_driving_results/cancer_dif2.joblib")
CANCER_F1   = joblib.load("cancer_driving_results/cancer_f1.joblib")
CANCER_DIF1 = joblib.load("cancer_driving_results/cancer_dif1.joblib")

# =========================================================
# HELPERS
# =========================================================
# =========================================================
# PREP DATA
# =========================================================
naive_pairs   = NetDesDuo.add_condition(NetDesDuo.prep_pairs(NAIVE_F2, NAIVE_DIF2), "Naive")
naive_singles = NetDesDuo.add_condition(NetDesDuo.prep_singles(NAIVE_F1, NAIVE_DIF1), "Naive")

cancer_pairs   = NetDesDuo.add_condition(NetDesDuo.prep_pairs(CANCER_F2, CANCER_DIF2), "Cancer")
cancer_singles = NetDesDuo.add_condition(NetDesDuo.prep_singles(CANCER_F1, CANCER_DIF1), "Cancer")

naive_all  = pd.concat([naive_pairs, naive_singles], ignore_index=True)
cancer_all = pd.concat([cancer_pairs, cancer_singles], ignore_index=True)
all_data   = pd.concat([naive_all, cancer_all], ignore_index=True)

# =========================================================
# GLOBAL AXIS LIMITS
# Use BOTH conditions together, then apply same limits to both panels
# Round outward to exact powers of 10
# =========================================================
x_min_raw = all_data["mse_f"].min()
x_max_raw = all_data["mse_f"].max()
y_min_raw = all_data["mse_dif"].min()
y_max_raw = all_data["mse_dif"].max()

x_lo = 10 ** np.floor(np.log10(x_min_raw))
x_hi = 10 ** np.ceil(np.log10(x_max_raw))
y_lo = 10 ** np.floor(np.log10(y_min_raw))
y_hi = 10 ** np.ceil(np.log10(y_max_raw))

print("X limits:", x_lo, "to", x_hi)
print("Y limits:", y_lo, "to", y_hi)

# =========================================================
# PLOT STYLING
# "Double as big"
# =========================================================
TITLE_FS  = 32
LABEL_FS  = 28
TICK_FS   = 24
LEGEND_FS = 24

PAIR_SIZE   = 110
SINGLE_SIZE = 150

PAIR_COLOR   = "tab:blue"
SINGLE_COLOR = "tab:orange"

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, condition_df, title in zip(
    axes,
    [naive_all, cancer_all],
    ["Naive", "Cancer"]
):
    pair_df = condition_df[condition_df["kind"] == "Pair"]
    single_df = condition_df[condition_df["kind"] == "Single"]

    ax.scatter(
        pair_df["mse_f"],
        pair_df["mse_dif"],
        s=PAIR_SIZE,
        color=PAIR_COLOR,
        marker="o",
        alpha=0.8,
        label="Pair"
    )

    ax.scatter(
        single_df["mse_f"],
        single_df["mse_dif"],
        s=SINGLE_SIZE,
        color=SINGLE_COLOR,
        marker="^",
        alpha=0.9,
        label="Single"
    )

    ax.set_xscale("log")
    ax.set_yscale("log")

    ax.set_xlim(x_lo, x_hi)
    ax.set_ylim(y_lo, y_hi)

    ax.set_title(title, fontsize=TITLE_FS)

    ax.set_xlabel("MSE: Forward vs Actual", fontsize=LABEL_FS)
    ax.set_ylabel("MSE: Forward vs Backward", fontsize=LABEL_FS)

    ax.tick_params(axis="both", which="major", labelsize=TICK_FS, length=9, width=1.5)
    ax.tick_params(axis="both", which="minor", length=5, width=1.2)

# One legend on the right
handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="center left",
    bbox_to_anchor=(0.91, 0.5),
    frameon=False,
    fontsize=LEGEND_FS,
    markerscale=1.4
)

fig.subplots_adjust(right=0.88, wspace=0.32)
plt.savefig("driving_naive_cancer_MSE.pdf", bbox_inches="tight")
plt.show()